In [3]:
# simulate_hierarchy.ipynb (content for a new Jupyter notebook)

import random
import time
import numpy as np
import pandas as pd
from SSELF_base import FootprintDatabase
from hierarchy_extension import HierarchicalSystem, HierarchicalCompany, Product

# Step 1: Define parameters
num_companies = 10
base_products = num_companies  # One product per company by default
companies_with_subs = ["Company_1", "Company_3"]
num_subs_per_company = 2

# ✅ Just use base_products:
system = HierarchicalSystem(num_companies=num_companies, num_products=base_products)

# Corrected starting ID (start right after base products)
starting_id = base_products + 1

# Step 3: Add hierarchies
system.add_hierarchies(
    target_company_names=companies_with_subs,
    num_subs_per_hierarchical=num_subs_per_company,
    starting_id=starting_id
)

# Step 4: Initialize footprint DB
footprint_db = FootprintDatabase(year=2024)

# ✅ Step 5: Register all products (including sub-company products)
for company in system.companies.values():
    for product in company.products:
        footprint_db.report(product.product_id, 0)
    if hasattr(company, "sub_companies"):
        for sub in company.sub_companies:
            for product in sub.products:
                footprint_db.report(product.product_id, 0)

# Step 6: Iterative update loop
updates_completed = False
iteration = 0
start_time = time.time()

for company in system.companies.values():
    company.num_updates = 0

while not updates_completed:
    iteration += 1
    print(f"\n--- Iteration {iteration} ---")
    any_company_updated = False

    for name, company in system.companies.items():
        previous = company.latest_update

        for sub in getattr(company, "sub_companies", []):
            sub.update_footprint(footprint_db)
            sub.report_footprint(footprint_db)

        company.update_footprint(footprint_db)
        new = company.latest_update

        if previous is None or not np.isclose(previous, new, atol=1e-6):
            print(f"  → {name} updated.")
            any_company_updated = True
            company.num_updates += 1
            company.report_footprint(footprint_db)

    updates_completed = not any_company_updated

end_time = time.time()

# Step 7: Summary
print("\n✅ Updates completed.")
print(f"⏱️ Time taken: {end_time - start_time:.2f} seconds")

print("\n📦 Final Footprints:")
for name, company in system.companies.items():
    print(f"{name}: {company.latest_update:.4f} kg CO2e/unit")
    for sub in getattr(company, "sub_companies", []):
        print(f"  {sub.name}: {sub.latest_update:.4f}")

print("\n🔁 Update Count:")
total_updates = sum(c.num_updates for c in system.companies.values())
for name, c in system.companies.items():
    print(f"{name}: {c.num_updates} updates")
print(f"Total updates: {total_updates}")

print("\n🗃️ Final footprint database:")
print(footprint_db.data.sort_values("id"))

# Optional: Additional forced update rounds
for i in range(3):
    print(f"\n--- Forced Update Iteration {i+1} ---")
    for name, company in system.companies.items():
        company.update_footprint(footprint_db)
        company.report_footprint(footprint_db)

print("\n🗃️ Final footprint database after forced iterations:")
print(footprint_db.data.sort_values("id"))



--- Iteration 1 ---

Calculating footprint for Company 1_Sub_1
  Product Product 11: Assigned footprint 0.0000
  Final footprint for Company 1_Sub_1: 0.0000

Calculating footprint for Company 1_Sub_2
  Product Product 12: Assigned footprint 0.0000
  Final footprint for Company 1_Sub_2: 0.0000

Calculating footprint for Company 1
  Product Product 1: Assigned footprint 31.3945
  Final footprint for Company 1: 31.3945
  → Company_1 updated.

Calculating footprint for Company 2
  Product Product 2: Assigned footprint 36.9029
  Final footprint for Company 2: 36.9029
  → Company_2 updated.

Calculating footprint for Company 3_Sub_1
  Product Product 13: Assigned footprint 0.0000
  Final footprint for Company 3_Sub_1: 0.0000

Calculating footprint for Company 3_Sub_2
  Product Product 14: Assigned footprint 0.3691
  Final footprint for Company 3_Sub_2: 0.3691

Calculating footprint for Company 3
  Product Product 3: Assigned footprint 9.2941
  Final footprint for Company 3: 9.2941
  → Compa

In [6]:
print("\n🔍 Carbon balance check:")

for name, company in system.companies.items():
    print(f"\n--- Balance check for {name} ---")

    # 1) Embodied impacts of purchases
    scores = footprint_db.data.set_index("id")["scores"].astype(float)
    in_emb = float(scores.loc[company.purchases.index].dot(company.purchases))

    # 2) Direct emissions
    in_direct = float(company.direct_impacts.sum().sum())

    # 3) Total carbon in
    carbon_in = in_emb + in_direct

    # 4) Carbon out (embodied in products sold)
    out_emb = 0
    for product in company.products:
        if product.product_id in company.sales.index:
            out_emb += float(product.footprint * company.sales.loc[product.product_id, "Sales"])

    # 5) Print check
    balanced = np.isclose(carbon_in, out_emb, atol=1e-6)

    print(f"  Purchases embodied (in_emb):     {in_emb:.6f}")
    print(f"  Direct emissions   (in_direct):  {in_direct:.6f}")
    print(f"  Total carbon in    (carbon_in):  {carbon_in:.6f}")
    print(f"  Sales embodied out (out_emb):    {out_emb:.6f}")
    print(f"  Balance?                         {balanced}")

    # Sub-companies
    if hasattr(company, "sub_companies"):
        for sub in company.sub_companies:
            print(f"\n  --- Sub-entity: {sub.name} ---")
            in_emb = float(scores.loc[sub.purchases.index].dot(sub.purchases)) if not sub.purchases.empty else 0
            in_direct = float(sub.direct_impacts.sum().sum())
            carbon_in = in_emb + in_direct

            out_emb = 0
            for product in sub.products:
                if product.product_id in sub.sales.index:
                    out_emb += float(product.footprint * sub.sales.loc[product.product_id, "Sales"])

            balanced = np.isclose(carbon_in, out_emb, rtol=1e-4)

            print(f"    Purchases embodied (in_emb):     {in_emb:.6f}")
            print(f"    Direct emissions   (in_direct):  {in_direct:.6f}")
            print(f"    Total carbon in    (carbon_in):  {carbon_in:.6f}")
            print(f"    Sales embodied out (out_emb):    {out_emb:.6f}")
            print(f"    Balance?                         {balanced}")



🔍 Carbon balance check:

--- Balance check for Company_1 ---
  Purchases embodied (in_emb):     453.757772
  Direct emissions   (in_direct):  418.000000
  Total carbon in    (carbon_in):  871.757772
  Sales embodied out (out_emb):    871.756988
  Balance?                         True

  --- Sub-entity: Company 1_Sub_1 ---
    Purchases embodied (in_emb):     54.277301
    Direct emissions   (in_direct):  0.000000
    Total carbon in    (carbon_in):  54.277301
    Sales embodied out (out_emb):    54.276601
    Balance?                         True

  --- Sub-entity: Company 1_Sub_2 ---
    Purchases embodied (in_emb):     104.398865
    Direct emissions   (in_direct):  0.000000
    Total carbon in    (carbon_in):  104.398865
    Sales embodied out (out_emb):    104.397549
    Balance?                         True

--- Balance check for Company_2 ---
  Purchases embodied (in_emb):     448.933991
  Direct emissions   (in_direct):  430.000000
  Total carbon in    (carbon_in):  878.933991


In [1]:
# simulate_hierarchy.ipynb (content for a new Jupyter notebook)

import random
import time
import numpy as np
import pandas as pd
from SSELF_base import FootprintDatabase
from hierarchy_extension import HierarchicalSystem, HierarchicalCompany, Product

# Step 1: Define parameters
num_companies = 10
base_products = num_companies  # One product per company by default
companies_with_subs = ["Company_1", "Company_3"]
num_subs_per_company = 2


# ✅ Just use base_products:
system = HierarchicalSystem(num_companies=num_companies, num_products=base_products)

# Corrected starting ID (start right after base products)
starting_id = base_products + 1

# Step 3: Add hierarchies
system.add_hierarchies(
    target_company_names=companies_with_subs,
    num_subs_per_hierarchical=num_subs_per_company,
    starting_id=starting_id
)

# Step 3: Add hierarchies with sub-companies and dynamic product IDs
system.add_hierarchies(
    target_company_names=companies_with_subs,
    num_subs_per_hierarchical=num_subs_per_company,
    starting_id = total_products + 1

)

# Step 4: Initialize footprint DB
footprint_db = FootprintDatabase(year=2024)

# ✅ Step 5: Register all products (including sub-company products)
for company in system.companies.values():
    for product in company.products:
        footprint_db.report(product.product_id, 0)
    if hasattr(company, "sub_companies"):
        for sub in company.sub_companies:
            for product in sub.products:
                footprint_db.report(product.product_id, 0)

# Step 6: Iterative update loop
updates_completed = False
iteration = 0
start_time = time.time()

for company in system.companies.values():
    company.num_updates = 0

while not updates_completed:
    iteration += 1
    print(f"\n--- Iteration {iteration} ---")
    any_company_updated = False

    for name, company in system.companies.items():
        previous = company.latest_update

        for sub in getattr(company, "sub_companies", []):
            sub.update_footprint(footprint_db)
            sub.report_footprint(footprint_db)

        company.update_footprint(footprint_db)
        new = company.latest_update

        if previous is None or not np.isclose(previous, new, atol=1e-6):
            print(f"  → {name} updated.")
            any_company_updated = True
            company.num_updates += 1
            company.report_footprint(footprint_db)

    updates_completed = not any_company_updated

end_time = time.time()

# Step 7: Summary
print("\n✅ Updates completed.")
print(f"⏱️ Time taken: {end_time - start_time:.2f} seconds")

print("\n📦 Final Footprints:")
for name, company in system.companies.items():
    print(f"{name}: {company.latest_update:.4f} kg CO2e/unit")
    for sub in getattr(company, "sub_companies", []):
        print(f"  {sub.name}: {sub.latest_update:.4f}")

print("\n🔁 Update Count:")
total_updates = sum(c.num_updates for c in system.companies.values())
for name, c in system.companies.items():
    print(f"{name}: {c.num_updates} updates")
print(f"Total updates: {total_updates}")

print("\n🗃️ Final footprint database:")
print(footprint_db.data.sort_values("id"))

# Optional: Additional forced update rounds
for i in range(3):
    print(f"\n--- Forced Update Iteration {i+1} ---")
    for name, company in system.companies.items():
        company.update_footprint(footprint_db)
        company.report_footprint(footprint_db)

print("\n🗃️ Final footprint database after forced iterations:")
print(footprint_db.data.sort_values("id"))


NameError: name 'total_products' is not defined

In [9]:
#Testing without hierarchy to make sure the base works
import time
import random
import numpy as np

# Step 1: Initialize footprint database with 0 values
for company in system.companies.values():
    for product in company.products:
        footprint_db_2024.report(product.product_id, 0)

# Step 2: Iterative update loop
updates_completed = False
iteration = 0

for company in system.companies.values():
    company.num_updates = 0  # Track how many times each company updated

start_time = time.time()

while not updates_completed:
    iteration += 1
    print(f"\n--- Iteration {iteration} ---")

    companies_to_check = list(system.companies.keys())
    companies_not_checked = companies_to_check[:]
    any_company_updated = False

    while companies_not_checked:
        random_company_name = random.choice(companies_not_checked)
        company = system.companies[random_company_name]

        print(f"Checking {random_company_name}")

        # Use existing footprint to recalculate and compare
        previous = company.latest_update
        company.update_footprint(footprint_db_2024)
        new = company.latest_update

        if previous is None or not np.isclose(previous, new, atol=1e-6):
            print(f"  -> Updated. Previous: {previous}, New: {new}")
            any_company_updated = True
            company.num_updates += 1
            company.report_footprint(footprint_db_2024)
        else:
            print("  -> No update needed.")

        companies_not_checked.remove(random_company_name)

    updates_completed = not any_company_updated

end_time = time.time()

# Step 3: Summary
print("\n✅ Updates completed.")
print(f"⏱️ Time taken: {end_time - start_time:.2f} seconds")

print("\n📦 Final Footprints:")
for name, company in system.companies.items():
    print(f"{name}: {company.latest_update:.4f} kg CO2e/unit")

print("\n🔁 Update Count:")
total_updates = sum(company.num_updates for company in system.companies.values())
for name, company in system.companies.items():
    print(f"{name}: {company.num_updates} updates")
print(f"Total updates: {total_updates}")

# Optional: Additional forced update rounds
for i in range(3):
    print(f"\n--- Forced Update Iteration {i+1} ---")
    for name, company in system.companies.items():
        company.update_footprint(footprint_db_2024)
        company.report_footprint(footprint_db_2024)



--- Iteration 1 ---
Checking Company_7

Calculating footprint for Company 7
  Product Product 7: Assigned footprint 32.5362
  Final footprint for Company 7: 32.5362
  -> Updated. Previous: 42.10108551239506, New: 32.536193065042376
Checking Company_4

Calculating footprint for Company 4
  Product Product 4: Assigned footprint 29.4908
  Final footprint for Company 4: 29.4908
  -> Updated. Previous: 26.713881665959896, New: 29.49077627107289
Checking Company_6

Calculating footprint for Company 6
  Product Product 6: Assigned footprint 50.2686
  Final footprint for Company 6: 50.2686
  -> Updated. Previous: 50.45252366948336, New: 50.26864666049353
Checking Company_9

Calculating footprint for Company 9
  Product Product 9: Assigned footprint 14.3162
  Final footprint for Company 9: 14.3162
  -> Updated. Previous: 18.846184889401027, New: 14.316245929330478
Checking Company_10

Calculating footprint for Company 10
  Product Product 10: Assigned footprint 44.3074
  Final footprint for Co

In [7]:
import os
print(os.getcwd())

C:\Users\marit.rognan\Documents\University\2nd paper\SSELF_python_code\company_hierarchy
